In [1]:
print("File loaded successfully.")

File loaded successfully.


In [2]:
import pandas as pd

In [4]:
cea = pd.read_csv("../raw/file_02.csv")

In [5]:
print("=" * 70)
print("SHAPE")
print("=" * 70)
print(cea.shape)

print("\n" + "=" * 70)
print("COLUMNS")
print("=" * 70)
print(cea.columns.tolist())

print("\n" + "=" * 70)
print("DATA TYPES")
print("=" * 70)
print(cea.dtypes)

print("\n" + "=" * 70)
print("FIRST 10 ROWS")
print("=" * 70)
print(cea.head(10))

print("\n" + "=" * 70)
print("LAST 10 ROWS")
print("=" * 70)
print(cea.tail(10))

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)
print(cea.isna().sum())

print("\n" + "=" * 70)
print("DUPLICATE ROWS")
print("=" * 70)
print("Duplicate rows:", cea.duplicated().sum())

print("\n" + "=" * 70)
print("UNIQUE VALUES PER COLUMN")
print("=" * 70)

for col in cea.columns:
    print(f"\n{col}:")
    print("  Unique:", cea[col].nunique(dropna=False))

    if cea[col].nunique(dropna=False) <= 50:
        print(cea[col].value_counts(dropna=False))

print("\n" + "=" * 70)
print("NUMERIC SUMMARY")
print("=" * 70)
print(cea.describe(include="all").T)

SHAPE
(4945, 9)

COLUMNS
['index', 'Date', 'Region', 'Thermal Generation Actual (in MU)', 'Thermal Generation Estimated (in MU)', 'Nuclear Generation Actual (in MU)', 'Nuclear Generation Estimated (in MU)', 'Hydro Generation Actual (in MU)', 'Hydro Generation Estimated (in MU)']

DATA TYPES
index                                     int64
Date                                        str
Region                                      str
Thermal Generation Actual (in MU)           str
Thermal Generation Estimated (in MU)        str
Nuclear Generation Actual (in MU)       float64
Nuclear Generation Estimated (in MU)    float64
Hydro Generation Actual (in MU)         float64
Hydro Generation Estimated (in MU)      float64
dtype: object

FIRST 10 ROWS
   index        Date        Region Thermal Generation Actual (in MU)  \
0      0  2017-09-01      Northern                            624.23   
1      1  2017-09-01       Western                          1,106.89   
2      2  2017-09-01      South

In [6]:
print("=" * 70)
print("NUCLEAR MISSINGNESS BY REGION")
print("=" * 70)

print(
    cea.groupby("Region")[
        [
            "Nuclear Generation Actual (in MU)",
            "Nuclear Generation Estimated (in MU)"
        ]
    ].apply(lambda x: x.isna().sum())
)

print("\n" + "=" * 70)
print("NUCLEAR AVAILABLE ROWS BY REGION")
print("=" * 70)

print(
    cea.groupby("Region")[
        "Nuclear Generation Actual (in MU)"
    ].apply(lambda x: x.notna().sum())
)

print("\n" + "=" * 70)
print("DATE RANGE")
print("=" * 70)

print("Start:", cea["Date"].min())
print("End:", cea["Date"].max())

print("\n" + "=" * 70)
print("ROWS PER REGION")
print("=" * 70)

print(cea["Region"].value_counts())

NUCLEAR MISSINGNESS BY REGION
              Nuclear Generation Actual (in MU)  \
Region                                            
Eastern                                     989   
NorthEastern                                989   
Northern                                      0   
Southern                                      0   
Western                                       0   

              Nuclear Generation Estimated (in MU)  
Region                                              
Eastern                                        989  
NorthEastern                                   989  
Northern                                         0  
Southern                                         0  
Western                                          0  

NUCLEAR AVAILABLE ROWS BY REGION
Region
Eastern           0
NorthEastern      0
Northern        989
Southern        989
Western         989
Name: Nuclear Generation Actual (in MU), dtype: int64

DATE RANGE
Start: 2017-09-01
End: 2020-08-01


In [7]:
print("=" * 70)
print("CONVERT THERMAL COLUMNS FOR INSPECTION")
print("=" * 70)

thermal_cols = [
    "Thermal Generation Actual (in MU)",
    "Thermal Generation Estimated (in MU)"
]

for col in thermal_cols:
    temp = pd.to_numeric(
        cea[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

    print(f"\n{col}")
    print("Failed conversions:", temp.isna().sum())
    print("Min:", temp.min())
    print("Max:", temp.max())

print("\n" + "=" * 70)
print("ACTUAL vs ESTIMATED DIFFERENCE")
print("=" * 70)

thermal_actual = pd.to_numeric(
    cea["Thermal Generation Actual (in MU)"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

thermal_estimated = pd.to_numeric(
    cea["Thermal Generation Estimated (in MU)"]
    .astype(str)
    .str.replace(",", "", regex=False)
)

thermal_diff = thermal_actual - thermal_estimated

print("Min difference:", thermal_diff.min())
print("Max difference:", thermal_diff.max())
print("Mean difference:", thermal_diff.mean())

print("\n" + "=" * 70)
print("ZERO / NEGATIVE VALUES")
print("=" * 70)

print("Thermal actual <= 0:", (thermal_actual <= 0).sum())
print("Thermal estimated <= 0:", (thermal_estimated <= 0).sum())
print("Hydro actual <= 0:", (cea["Hydro Generation Actual (in MU)"] <= 0).sum())
print("Hydro estimated <= 0:", (cea["Hydro Generation Estimated (in MU)"] <= 0).sum())

CONVERT THERMAL COLUMNS FOR INSPECTION

Thermal Generation Actual (in MU)
Failed conversions: 0
Min: 12.34
Max: 1395.97

Thermal Generation Estimated (in MU)
Failed conversions: 0
Min: 12.38
Max: 1442.38

ACTUAL vs ESTIMATED DIFFERENCE
Min difference: -238.17000000000007
Max difference: 434.97
Mean difference: 28.58324165824065

ZERO / NEGATIVE VALUES
Thermal actual <= 0: 0
Thermal estimated <= 0: 0
Hydro actual <= 0: 2
Hydro estimated <= 0: 2


In [8]:
print("=" * 70)
print("HYDRO ZERO-VALUE ROWS")
print("=" * 70)

print(
    cea[
        (cea["Hydro Generation Actual (in MU)"] <= 0) |
        (cea["Hydro Generation Estimated (in MU)"] <= 0)
    ][
        [
            "Date",
            "Region",
            "Hydro Generation Actual (in MU)",
            "Hydro Generation Estimated (in MU)"
        ]
    ]
)

HYDRO ZERO-VALUE ROWS
           Date        Region  Hydro Generation Actual (in MU)  \
107  2017-09-22      Southern                              0.0   
109  2017-09-22  NorthEastern                              0.0   

     Hydro Generation Estimated (in MU)  
107                                 0.0  
109                                 0.0  


In [9]:
print("\n" + "=" * 70)
print("NEGATIVE VALUES")
print("=" * 70)

numeric_generation_cols = [
    "Nuclear Generation Actual (in MU)",
    "Nuclear Generation Estimated (in MU)",
    "Hydro Generation Actual (in MU)",
    "Hydro Generation Estimated (in MU)"
]

for col in numeric_generation_cols:
    print(
        f"{col}:",
        (cea[col] < 0).sum()
    )


NEGATIVE VALUES
Nuclear Generation Actual (in MU): 0
Nuclear Generation Estimated (in MU): 0
Hydro Generation Actual (in MU): 0
Hydro Generation Estimated (in MU): 0


In [10]:
cea_clean = cea.copy()

# Remove unnecessary index column
cea_clean = cea_clean.drop(columns=["index"])

# Parse date
cea_clean["Date"] = pd.to_datetime(cea_clean["Date"])

# Convert thermal generation from formatted strings to numeric
thermal_cols = [
    "Thermal Generation Actual (in MU)",
    "Thermal Generation Estimated (in MU)"
]

for col in thermal_cols:
    cea_clean[col] = pd.to_numeric(
        cea_clean[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Validate
print("Shape:", cea_clean.shape)

print("\nData types:")
print(cea_clean.dtypes)

print("\nMissing values:")
print(cea_clean.isna().sum())

print("\nDuplicate Date-Region rows:")
print(
    cea_clean.duplicated(
        subset=["Date", "Region"]
    ).sum()
)

print("\nDate range:")
print(cea_clean["Date"].min(), "to", cea_clean["Date"].max())

Shape: (4945, 8)

Data types:
Date                                    datetime64[us]
Region                                             str
Thermal Generation Actual (in MU)              float64
Thermal Generation Estimated (in MU)           float64
Nuclear Generation Actual (in MU)              float64
Nuclear Generation Estimated (in MU)           float64
Hydro Generation Actual (in MU)                float64
Hydro Generation Estimated (in MU)             float64
dtype: object

Missing values:
Date                                       0
Region                                     0
Thermal Generation Actual (in MU)          0
Thermal Generation Estimated (in MU)       0
Nuclear Generation Actual (in MU)       1978
Nuclear Generation Estimated (in MU)    1978
Hydro Generation Actual (in MU)            0
Hydro Generation Estimated (in MU)         0
dtype: int64

Duplicate Date-Region rows:
0

Date range:
2017-09-01 00:00:00 to 2020-08-01 00:00:00


In [12]:
cea_clean.to_csv(
    "../clean/fact_regional_generation_daily.csv",
    index=False,
    encoding="utf-8"
)

print("Saved:", "../clean/fact_regional_generation_daily.csv")
print("Shape:", cea_clean.shape)

Saved: ../clean/fact_regional_generation_daily.csv
Shape: (4945, 8)


In [13]:
# ============================================================
# CEA REGIONAL GENERATION FEATURES
# ============================================================

import numpy as np
import pandas as pd

cea_feat = cea_clean.copy()

# ------------------------------------------------------------
# 1. Actual vs Estimated deviation
# ------------------------------------------------------------

cea_feat["thermal_actual_estimated_diff"] = (
    cea_feat["Thermal Generation Actual (in MU)"]
    - cea_feat["Thermal Generation Estimated (in MU)"]
)

cea_feat["hydro_actual_estimated_diff"] = (
    cea_feat["Hydro Generation Actual (in MU)"]
    - cea_feat["Hydro Generation Estimated (in MU)"]
)

cea_feat["nuclear_actual_estimated_diff"] = (
    cea_feat["Nuclear Generation Actual (in MU)"]
    - cea_feat["Nuclear Generation Estimated (in MU)"]
)

# ------------------------------------------------------------
# 2. Aggregate by region
# ------------------------------------------------------------

regional_features = (
    cea_feat
    .groupby("Region")
    .agg(
        thermal_generation_mean_mu=(
            "Thermal Generation Actual (in MU)", "mean"
        ),
        thermal_generation_std_mu=(
            "Thermal Generation Actual (in MU)", "std"
        ),

        hydro_generation_mean_mu=(
            "Hydro Generation Actual (in MU)", "mean"
        ),
        hydro_generation_std_mu=(
            "Hydro Generation Actual (in MU)", "std"
        ),

        nuclear_generation_mean_mu=(
            "Nuclear Generation Actual (in MU)", "mean"
        ),
        nuclear_generation_std_mu=(
            "Nuclear Generation Actual (in MU)", "std"
        ),

        thermal_estimated_diff_mean_mu=(
            "thermal_actual_estimated_diff", "mean"
        ),
        hydro_estimated_diff_mean_mu=(
            "hydro_actual_estimated_diff", "mean"
        ),
        nuclear_estimated_diff_mean_mu=(
            "nuclear_actual_estimated_diff", "mean"
        ),

        days_available=(
            "Date", "nunique"
        )
    )
    .reset_index()
)

print("Shape:", regional_features.shape)

print("\nColumns:")
print(regional_features.columns.tolist())

print("\nRegional features:")
print(regional_features)

Shape: (5, 11)

Columns:
['Region', 'thermal_generation_mean_mu', 'thermal_generation_std_mu', 'hydro_generation_mean_mu', 'hydro_generation_std_mu', 'nuclear_generation_mean_mu', 'nuclear_generation_std_mu', 'thermal_estimated_diff_mean_mu', 'hydro_estimated_diff_mean_mu', 'nuclear_estimated_diff_mean_mu', 'days_available']

Regional features:
         Region  thermal_generation_mean_mu  thermal_generation_std_mu  \
0       Eastern                  487.486067                  28.291007   
1  NorthEastern                   32.472993                   2.652038   
2      Northern                  662.333933                  39.295874   
3      Southern                  617.546572                  41.680935   
4       Western                 1220.052224                  89.116100   

   hydro_generation_mean_mu  hydro_generation_std_mu  \
0                 49.228129                23.207204   
1                 17.808210                 8.391261   
2                190.954661             

In [15]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------- ----------- 6.0/8.4 MB 31.8 MB/s eta 0:00:01
   ---------------------------------------- 8.4/8.4 MB 28.4 MB/s  0:00:00
   ---------------------------------------- 0.0/37.4 MB ? eta -:--:--
   ----- ---------------------------------- 4.7/37.4 MB 23.8 MB/s eta 0:00:02
   -------- ------------------------------- 7.9/37.4 MB 18.1 MB/s eta 0:00:02
   ------------ --------------------------- 11.5/37.4 MB 18.3 MB/s eta 0:00:02
   --------------- ------------------------ 14.7/37.4 MB 17.8 MB/s eta 0:00:02
   -------------------- ------------------- 18.9/37.4 MB 18.0 MB/s eta 0:00:02
   ------------------------ --------------- 22.5/37.4 MB 18.0 MB/s eta 0:00:01
   ------------------------------- -------- 29.4/37.4 MB 20.0 MB/s eta 0:00:01
   ---------------------------------------  36.4/37.4 MB 21.6 MB/s eta 0:00:01
   ---------------------------------------- 37.4/37.4 MB 20.7 MB/s  0:00:01

   


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from scipy.stats import linregress

def calculate_trend(group, value_col):
    data = group[["Date", value_col]].dropna()

    if len(data) < 2:
        return np.nan

    x = (data["Date"] - data["Date"].min()).dt.days
    y = data[value_col]

    return linregress(x, y).slope


trend_rows = []

for region, group in cea_feat.groupby("Region"):
    trend_rows.append({
        "Region": region,

        "thermal_generation_trend_mu_per_day":
            calculate_trend(
                group,
                "Thermal Generation Actual (in MU)"
            ),

        "hydro_generation_trend_mu_per_day":
            calculate_trend(
                group,
                "Hydro Generation Actual (in MU)"
            ),

        "nuclear_generation_trend_mu_per_day":
            calculate_trend(
                group,
                "Nuclear Generation Actual (in MU)"
            )
    })

regional_trends = pd.DataFrame(trend_rows)

regional_features = regional_features.merge(
    regional_trends,
    on="Region",
    validate="one_to_one"
)

print("Shape:", regional_features.shape)
print(regional_features)

Shape: (5, 14)
         Region  thermal_generation_mean_mu  thermal_generation_std_mu  \
0       Eastern                  487.486067                  28.291007   
1  NorthEastern                   32.472993                   2.652038   
2      Northern                  662.333933                  39.295874   
3      Southern                  617.546572                  41.680935   
4       Western                 1220.052224                  89.116100   

   hydro_generation_mean_mu  hydro_generation_std_mu  \
0                 49.228129                23.207204   
1                 17.808210                 8.391261   
2                190.954661                88.972237   
3                 71.900243                15.587556   
4                 36.638362                11.219053   

   nuclear_generation_mean_mu  nuclear_generation_std_mu  \
0                         NaN                        NaN   
1                         NaN                        NaN   
2                   27.

In [18]:
regional_features.to_csv(
    "../clean/fact_regional_generation_features.csv",
    index=False,
    encoding="utf-8"
)

print("Saved successfully.")
print("Shape:", regional_features.shape)

Saved successfully.
Shape: (5, 14)
